<a href="https://colab.research.google.com/github/Naveena-g/Banking_risk_analysis/blob/code_part/capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 1: Data aquisition and loading**

 ## Task 1: Setup Snowflake Environment.




• Create warehouse, database, schema.
• Setup roles, users, and apply least-privilege access.
• Configure resource monitors to control compute cost.

## Task 2: Load Flat Files via SnowSQL


---




• Use PUT and COPY INTO commands for:
 o Customer_Master.csv
 o Region_Hierarchy.csv
 o Collateral_Register.csv
• Define staging tables with data-type mapping and validation rules.


In [ ]:
-----------------BUREAU_SCORES------------------------
create or replace table Bureau_Scores(
BureauID int,
CustomerID int,
Score int,
Rating	string,
ReportMonth string);

put file://C:\Users\user\Downloads\Bureau_Scores.csv @~;

copy into Bureau_Scores
from @~
file_format=(type='csv'
field_delimiter=','
skip_header=1);

select*from BUREAU_SCORES;

## Task 3: Core Data Ingestion via IICS



• Extract data from Loan_Accounts (core system).
• Transform and load into stg_loan_accounts in Snowflake.
• Schedule refresh using IICS Taskflow (daily).

## Task 4: Real-Time Collection Feed via Snowpipe



• Create External Stage linked to S3 bucket.
• Define File Format (CSV) with auto-date detection.
• Build and enable Pipe for Collections_Events.csv ingestion.
• Validate ingestion by simulating new file drops into S3.

In [ ]:
use role accountadmin;

CREATE OR REPLACE STORAGE INTEGRATION my_s3_integration
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = S3
  ENABLED = TRUE
  STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::049903362104:role/capstone_role'
  STORAGE_ALLOWED_LOCATIONS = ('s3://s3-bucket-parth-dange1/capstone_data');


desc integration my_s3_integration;

create or replace stage pipe_stage
URL = 's3://s3-bucket-parth-dange1/capstone_data/'
storage_integration = my_s3_integration
file_format = my_csv_format;


create or replace table collections_events(
EventID number,
LoanID number,
EventDate date,
DPD	int,
ActionType string,
PTP_Flag int,
PaymentAmount decimal(10,2),
AgentID int
);

alter PIPE collections_events_pipe refresh;
  AUTO_INGEST = TRUE
  AS
  COPY INTO collections_events
  FROM @pipe_stage
  FILE_FORMAT = (TYPE = 'CSV')
  ON_ERROR = 'CONTINUE';

describe stage pipe_stage;

show pipes;

select system$pipe_status('collections_events_pipe');

select * from collections_events;

select $1, $2, $3, $4 from @pipe_stage;

CREATE or replace TABLE CUSTOMERS (
    Order_ID STRING,  -- For alphanumeric Order IDs
    Order_Date string,
    Ship_Date string,
    Customer_Name varchar,
    Segment STRING,
    Country STRING,
    Region STRING,
    Category STRING,
    Sub_Category STRING,
    Sales FLOAT,  -- Sales values are numeric (decimals)
    Quantity INT,  -- Quantity is an integer
    Profit FLOAT   -- Profit is numeric (decimals)
);



select * from customers;


CREATE OR REPLACE PIPE collections_events_pipe
  AUTO_INGEST = TRUE
  AS
  COPY INTO collections_events
  FROM @pipe_stage
  FILE_FORMAT = (TYPE = 'CSV')
  ON_ERROR = 'CONTINUE';




select $1, $2, $3, $5, $6, $4, $7, $8, $9, $10 from @collections_events_pipe;


show pipes;

select * from collections_events;

select $1 from @pipe_stage;

select system$pipe_status('collections_events_pipe');





show tables;







# **PHASE 2: 🏗️ Data Modeling and Star Schema**

## Task 5: Build Dimension Tables



• DimCustomer: CustomerID, Name (masked), Segment, RiskBand, Region.
• DimProduct: ProductID, LoanType, Tenor, APRTier.
• DimRegion: Country, Zone, Branch, RegionHierarchy.
• DimCollateral: CollateralID, CollateralType, ValuationBand.

In [ ]:
#Dim_cusotmers

CREATE TABLE DIMCUSTOMER (
    CustomerSK   NUMBER AUTOINCREMENT PRIMARY KEY,   -- Surrogate key
    CustomerID   VARCHAR(50) UNIQUE,
    CustomerName VARCHAR(500),
    Segment      VARCHAR(50),
    RiskBand     VARCHAR(10),
    RegionID     VARCHAR(50)
);

MERGE INTO DIMCUSTOMER tgt
USING (
    SELECT
        CustomerID,
        CustomerName,
        CASE
            WHEN AnnualIncome < 30000 THEN 'Value Customers'
            WHEN AnnualIncome < 100000 THEN 'Priority Customers'
            ELSE 'Premier Customers'
        END AS Segment,
        RiskBand,
        RegionID
    FROM CUSTOMER_MASTER
) src
ON tgt.CustomerID = src.CustomerID

WHEN MATCHED AND (
        tgt.CustomerName <> src.CustomerName
        OR tgt.Segment <> src.Segment
        OR tgt.RiskBand <> src.RiskBand
        OR tgt.RegionID <> src.RegionID
    )
THEN UPDATE SET
        CustomerName = src.CustomerName,
        Segment = src.Segment,
        RiskBand = src.RiskBand,
        RegionID = src.RegionID

WHEN NOT MATCHED THEN
    INSERT (CustomerID, CustomerName, Segment, RiskBand, RegionID)
    VALUES (src.CustomerID, src.CustomerName, src.Segment, src.RiskBand, src.RegionID);

select * from dimcustomer;




In [ ]:
#dim_region
CREATE OR REPLACE TABLE DIMREGION (
    RegionSK        NUMBER,                -- Surrogate Key
    RegionID        NUMBER(38,0) UNIQUE,   -- Natural/Source Key
    Country         VARCHAR(100) NOT NULL,
    Zone            VARCHAR(100)g,
    Branch          VARCHAR(100),
    RegionHierarchy VARCHAR(500)
);
select * from collection_events;

-- Step 1: Create table (RegionSK will be populated later via ROW_NUMBER)
CREATE OR REPLACE TABLE DIMREGION (
    RegionSK        NUMBER,                -- Surrogate Key
    RegionID        NUMBER(38,0) UNIQUE,   -- Natural/Source Key
    Country         VARCHAR(100) NOT NULL,
    Zone            VARCHAR(100),
    Branch          VARCHAR(100),
    RegionHierarchy VARCHAR(500)
);

-- Step 2: Merge new/updated data
MERGE INTO DIMREGION tgt
USING (
    SELECT
        REGIONID,
        COUNTRY,
        ZONE,
        BRANCH,
        TRIM(COUNTRY) || ' > ' || TRIM(ZONE) || ' > ' || TRIM(BRANCH) AS REGIONHIERARCHY
    FROM REGION_HIERARCHY
) src
ON tgt.RegionID = src.REGIONID
WHEN MATCHED AND tgt.RegionHierarchy <> src.REGIONHIERARCHY
    THEN UPDATE SET
        Country = src.COUNTRY,
        Zone = src.ZONE,
        Branch = src.BRANCH,
        RegionHierarchy = src.REGIONHIERARCHY
WHEN NOT MATCHED
    THEN INSERT (RegionID, Country, Zone, Branch, RegionHierarchy)
         VALUES (src.REGIONID, src.COUNTRY, src.ZONE, src.BRANCH, src.REGIONHIERARCHY);

-- Step 3: Backfill RegionSK for rows where it is NULL
UPDATE DIMREGION
SET RegionSK = seq
FROM (
    SELECT RegionID, ROW_NUMBER() OVER (ORDER BY RegionID) AS seq
    FROM DIMREGION
) t
WHERE DIMREGION.RegionID = t.RegionID
  AND DIMREGION.RegionSK IS NULL;

-- Step 4: Verify
SELECT * FROM DIMREGION
ORDER BY RegionSK;


         SELECT * FROM DIMREGION;




In [ ]:
#dim_collateral

-- 1. Drop table if exists (for recreation)
DROP TABLE IF EXISTS DimCollateral;

-- 2. Create DimCollateral table with surrogate key
CREATE OR REPLACE TABLE DimCollateral (
    CollateralSk INT AUTOINCREMENT PRIMARY KEY,  -- Surrogate Key
    CollateralID INT,                            -- Natural Key
    LoanID INT,                                  -- Associated Loan
    CollateralType VARCHAR(50),
    ValuationBand VARCHAR(20)
);

-- 3. Insert data from source table with ValuationBand
INSERT INTO DimCollateral (CollateralID, LoanID, CollateralType, ValuationBand)
SELECT DISTINCT
    CollateralID,
    LoanID,
    CollateralType,
    CASE
        WHEN Valuation < 50000 THEN 'Low'
        WHEN Valuation BETWEEN 50000 AND 200000 THEN 'Medium'
        ELSE 'High'
    END AS ValuationBand
FROM collateral_register
ORDER BY CollateralID;

-- 4. Verify the table
SELECT * FROM DimCollateral;

-- 5. Optional: Describe table structure
DESC TABLE DimCollateral;



In [ ]:
#dim_product

select * from NV_LAON_ACC;
TRUNCATE TABLE DIMPRODUCT;


drop table dimproduct;
CREATE TABLE DIMPRODUCT (
    ProductSK   NUMBER AUTOINCREMENT PRIMARY KEY,   -- Surrogate key
    ProductID   VARCHAR(50) UNIQUE,                 -- Natural / Source key
    LoanType    VARCHAR(100),
    APR         NUMBER,
    TenorMonths NUMBER
);

MERGE INTO DIMPRODUCT tgt
USING (
    SELECT
        ProductID,
        APR,
        TenorMonths,
        CASE
            WHEN TenorMonths < 24 THEN 'Near Term Loan'
            WHEN TenorMonths BETWEEN 25 AND 150 THEN 'Core Growth Loan'
            ELSE 'Strategic Long Term Loan'
        END AS LoanType
    FROM  NV_LAON_ACC
) src
ON tgt.ProductID = src.ProductID

WHEN MATCHED AND (
       tgt.LoanType <> src.LoanType
    OR tgt.APR <> src.APR
    OR tgt.TenorMonths <> src.TenorMonths
)
THEN UPDATE SET
        LoanType    = src.LoanType,
        APR         = src.APR,
        TenorMonths = src.TenorMonths

WHEN NOT MATCHED THEN
    INSERT (ProductID, LoanType, APR, TenorMonths)
    VALUES (src.ProductID, src.LoanType, src.APR, src.TenorMonths);

SELECT * FROM DIMPRODUCT;



## Task 6: Build Fact Tables


• FactLoan: LoanID, OriginationAmount, APR, Tenor, CurrentBalance, joins dimensions.
• FactCollections: DPD, ContactResult, PTP, Cure%, joins with customer and region dimensions.
• FactBureau: BureauScore, Rating, Month, joins with DimCustomer.

In [ ]:
-----------------------FactBureau---------------------------

CREATE OR REPLACE TABLE FactBureau (
    CustomerSK  INT NOT NULL,              -- FK from DimCustomer
    BureauScore   INT,
    Rating        VARCHAR(50),
    Month         DATE,
    FOREIGN KEY (CustomerSK) REFERENCES DimCustomer(CustomerSK)
);


MERGE INTO FactBureau AS target
USING (
    SELECT
        d.CustomerSK,
        b.Score AS BureauScore,
        b.Rating,
        b.ReportMonth AS Month
    FROM BUREAU_SCORES b
    LEFT JOIN DimCustomer d
        ON b.CustomerID = d.CustomerID
) AS source
ON (target.CustomerSK = source.CustomerSK)
WHEN MATCHED THEN
    UPDATE SET
        target.BureauScore = source.BureauScore,
        target.Rating = source.Rating
WHEN NOT MATCHED THEN
    INSERT (CustomerSK, BureauScore, Rating, Month)
    VALUES (source.CustomerSK, source.BureauScore, source.Rating, source.Month);

select * from factbureau;


In [ ]:
# fact collection

# create factcollections table query
CREATE OR REPLACE TABLE factcollections (
    CustomerSK    NUMBER,
    RegionSK      NUMBER,
    DPD           NUMBER(18,2),
    ContactResult VARCHAR(100),
    PTP           NUMBER(18,2),
    CurePercent   NUMBER(5,2),

    -- Foreign Key Constraints
    CONSTRAINT fk_customer FOREIGN KEY (CustomerSK)
        REFERENCES dimcustomer(CustomerSK),
    CONSTRAINT fk_region FOREIGN KEY (RegionSK)
        REFERENCES dimregion(RegionSK)
);

# loading data query

MERGE INTO factcollections tgt
USING (
    SELECT
        dc.CustomerSK,
        dr.RegionSK,
        ce.DPD,

        CASE
            WHEN COALESCE(ce.PaymentAmount,0) > 0 THEN 'Paid'
            WHEN COALESCE(ce.PTP_Flag,0) = 1 AND COALESCE(ce.PaymentAmount,0) = 0 THEN 'Promise to Pay'
            WHEN COALESCE(ce.PaymentAmount,0) = 0 AND COALESCE(ce.PTP_Flag,0) = 0 THEN 'Not Connected'
            ELSE 'No Action'
        END AS ContactResult,

        ce.PTP_Flag AS PTP,

        ROUND(
            LEAST(
                (SUM(ce.PaymentAmount) OVER (PARTITION BY ld.CustomerID) /
                 NULLIF(SUM(ld.CurrentBalance) OVER (PARTITION BY ld.CustomerID),0)) * 100,
                100
            ), 2
        ) AS CurePercent

    FROM collection_events ce
    LEFT JOIN loan_data ld
        ON ce.LoanID = ld.LoanID
    LEFT JOIN dimcustomer dc
        ON ld.CustomerID = dc.CustomerID
    LEFT JOIN dimregion dr
        ON ld.RegionID = dr.RegionID
) src
ON tgt.CustomerSK = src.CustomerSK
   AND tgt.RegionSK = src.RegionSK
   AND tgt.DPD = src.DPD
WHEN MATCHED THEN
    UPDATE SET
        ContactResult = src.ContactResult,
        PTP = src.PTP,
        CurePercent = src.CurePercent
WHEN NOT MATCHED THEN
    INSERT (CustomerSK, RegionSK, DPD, ContactResult, PTP, CurePercent)
    VALUES (src.CustomerSK, src.RegionSK, src.DPD, src.ContactResult, src.PTP, src.CurePercent);



## Task 7: Apply Surrogate Keys and Constraints


• Add surrogate keys to all dimension tables.
• Apply foreign key constraints and use MERGE for incremental upserts.

# **PHASE 3: ⚙️ Incremental Automation Using Streams, Tasks, and MVs**



## Task 8: Stream Setup



• Create STREAM on stg_loan_accounts and stg_collections_events.
• Track changes with METADATA$ACTION to detect inserts/updates.

In [ ]:
# stream 1 loan_data

CREATE OR REPLACE STREAM loan_accounts_stream
ON TABLE loan_data
SHOW_INITIAL_ROWS = FALSE;
select * from loan_data;

INSERT INTO loan_data (
  LOANID, CUSTOMERID, PRODUCTID, REGIONID, ORIGINATIONDATE, APR, TENORMONTHS,
  ORIGINATIONAMOUNT, CURRENTBALANCE, STATUS, INTERESTTYPE, REPAYMENTFREQUENCY,
  RISKBAND, CHANNEL, COLLATERALIZEDFLAG, WRITEOFFFLAG, CLOSEDDATE, LASTPAYMENTDATE, CURRENCY
) VALUES
(9000000, 107270, 511, 3124, '2022-07-20', 26.8, 9, 1619.22, 9.45, 'Closed', 'Fixed', 'Biweekly', 'A', 'Partner', 0, 0, '2025-08-08', '2025-08-08', 'USD'),
(9000001, 100860, 507, 3134, '2024-03-24', 11.84, 14, 4187.09, 32.81, 'Closed', 'Variable', 'Monthly', 'B', 'Branch', 0, 0, '2025-09-14', '2025-09-14', 'GBP');


UPDATE loan_data
SET CURRENTBALANCE = 250.00,
    STATUS = 'Active'
WHERE LOANID = 9000000;

UPDATE loan_data
SET APR = 12.50,
    CHANNEL = 'Partner'
WHERE LOANID = 9000001;

-- DELETE example (will produce DELETE row in stream)
DELETE FROM loan_data
WHERE LOANID = 9000001;


select * from loan_data;
select * from loan_accounts_stream;


# stream 2 collection_events

CREATE OR REPLACE STREAM collections_events_stream
ON TABLE collection_events
SHOW_INITIAL_ROWS = FALSE;

-- view current table rows
SELECT * FROM collection_events;

-- INSERT sample rows (includes the two you provided + 3 more)
INSERT INTO collection_events (EventID, LoanID, EventDate, DPD, ActionType, PTP_Flag, PaymentAmount, AgentID) VALUES
(20000000, 9016796, '2020-10-25', 30, 'Call', 1, 859.75, 1034),
(20000001, 9002110, '2022-10-14', 30, 'SMS', 0, 0.00, 1041),
(20000002, 9011001, '2023-01-12', 15, 'Call', 1, 450.25, 1022),
(20000003, 9013005, '2023-02-05', 45, 'SMS', 0, 0.00, 1031),
(20000004, 9022200, '2023-03-18', 60, 'Visit', 1, 1200.50, 1055);

-- UPDATE examples (produces UPDATE change rows in stream: DELETE old image + INSERT new image with METADATA$ISUPDATE = TRUE)
UPDATE collection_events
SET PaymentAmount = 900.00
WHERE EventID = 20000000;

UPDATE collection_events
SET ActionType = 'Visit'
WHERE EventID = 20000003;

-- DELETE examples (produces DELETE rows in stream)
DELETE FROM collection_events
WHERE EventID = 20000001;

DELETE FROM collection_events
WHERE EventID = 20000004;

-- view table after DML
SELECT * FROM collection_events;

-- view stream contents with metadata to detect INSERT/UPDATE/DELETE
SELECT
  EventID, LoanID, EventDate, DPD, ActionType, PTP_Flag, PaymentAmount, AgentID,
  METADATA$ACTION   AS action_type,
  METADATA$ISUPDATE AS is_update,
  METADATA$ROW_ID   AS row_id,
FROM collections_events_stream;



## Task 9: Task Automation


• Define hourly TASK chains to:
 o Insert new loans into FactLoan.
 o Update latest DPD and recovery data in FactCollections.
 o Record pipeline metadata in audit_log.


**FactLoan task**

In [ ]:
# FactLoan task code

In [ ]:
# create dim
CREATE or replace TABLE DimCustomer_backup (
    CustomerSK INT PRIMARY KEY,
    CustomerID int NOT NULL UNIQUE,
    customerName VARCHAR(100),  -- masked name
    Segment VARCHAR(50),
    RiskBand VARCHAR(20),
    RegionID int,phone string,email string
);

CREATE or replace TABLE DIMPRODUCT_BACKUP (
    ProductSK INT PRIMARY KEY,
    ProductID int NOT NULL UNIQUE,
    LoanType VARCHAR(50),
    APRTier VARCHAR(20),Tenor varchar(100)
);


CREATE or replace TABLE DIMREGION_BACKUP (
    RegionSK INT  PRIMARY KEY,regionid int not null unique,
    Country VARCHAR(50),
    Zone VARCHAR(50),
    Branch VARCHAR(50),
    RegionHierarchy VARCHAR(100)
);

CREATE or replace TABLE DIMCOLLATERAL_BACKUP (
    CollateralSK INT  PRIMARY KEY,
    CollateralID int NOT NULL UNIQUE,
    CollateralType VARCHAR(50),
    ValuationBand VARCHAR(20)
);


alter task TASK_MERGE_DIMCOLLATERAL_HOURLY resume;
alter task TASK_MERGE_DIMPRODUCT_HOURLY resume;
alter task TASK_MERGE_DIMREGION_HOURLY resume;
alter task TASK_MERGE_DImcustomer_HOURLY resume;
alter task TASK_MERGE_DIMCOLLATERAL_HOURLY suspend;
alter task TASK_MERGE_DIMPRODUCT_HOURLY suspend;
alter task TASK_MERGE_DIMREGION_HOURLY suspend;
alter task TASK_MERGE_DImcustomer_HOURLY suspend;

In [ ]:
# create task_merge_dimcollateral_hourly



-- collateral
CREATE OR REPLACE TASK task_merge_dimcollateral_hourly
  WAREHOUSE = my_warehouse
  SCHEDULE = "1 MINUTE"
AS
MERGE INTO DIMCOLLATERAL_BACKUP AS target
USING (
    SELECT
        row_number() OVER (ORDER BY collateralid) AS collateralSK,
        COLLATERALID,
        COLLATERALTYPE,
        CASE
            WHEN VALUATION < 10000 THEN 'Low'
            WHEN VALUATION <= 50000 THEN 'Medium'
            ELSE 'High'
        END AS ValuationBand
    FROM dummycollateralregister
) AS source
ON target.CollateralID = source.CollateralID

-- Update the record if the match is found
WHEN MATCHED THEN
    UPDATE SET
        target.CollateralType = source.CollateralType,
        target.ValuationBand = source.ValuationBand

-- Insert the record if no match is found
WHEN NOT MATCHED THEN
    INSERT
        (collateralsk, CollateralID, CollateralType, ValuationBand)
    VALUES
        (source.collateralSK, source.CollateralID, source.CollateralType, source.ValuationBand);



In [ ]:
#task_merge_dimregion_hourly
desc table dimregion_backup;
CREATE OR REPLACE TASK task_merge_dimregion_hourly
  WAREHOUSE = my_warehouse
  SCHEDULE = "1 MINUTE"
AS
MERGE INTO dimregion_backup AS target
USING (
    SELECT
        ROW_NUMBER() OVER (ORDER BY regionid) AS regionsk,
        regionid,
        country,
        zone,
        branch,
        CONCAT_WS('>', country, zone, branch) AS regionhierarchy
    FROM dummyregionhierarchy
) AS src
ON target.regionid = src.regionid

-- If the data already exists, update only when there's a change
WHEN MATCHED AND (
    target.country != src.country
    OR target.zone != src.zone
    OR target.branch != src.branch
) THEN
    UPDATE SET
        target.country = src.country,
        target.zone = src.zone,
        target.branch = src.branch,
        target.regionhierarchy = src.regionhierarchy

-- If the data does not exist, insert new records
WHEN NOT MATCHED THEN
    INSERT (regionsk, regionid, country, zone, branch, regionhierarchy)
    VALUES (src.regionsk, src.regionid, src.country, src.zone, src.branch, src.regionhierarchy);



In [ ]:
# create task_merge_dimcustomer_hourly


select * from dimregion_backup;


-- customer
CREATE OR REPLACE TASK task_merge_dimcustomer_hourly
  WAREHOUSE = my_warehouse
  SCHEDULE = "1 minutes"
AS
MERGE INTO DimCustomer_backup AS target
USING (
    SELECT
        ROW_NUMBER() OVER (ORDER BY customerid) AS customersk,  -- Surrogate key generation
        CustomerID,
        CustomerName,
        Email,
        Phone,
        Age,
        AnnualIncome,
        RiskBand,
        RegionID,

        CASE
            WHEN (RiskBand IN ('A', 'B') AND AnnualIncome > 1000000) THEN 'Priority Customer'
            WHEN (RiskBand IN ('B', 'C') AND AnnualIncome BETWEEN 100000 AND 300000) THEN 'Premier Customer'
            WHEN (RiskBand IN ('D', 'E', 'F') AND AnnualIncome < 100000) THEN 'Value Customer'
            ELSE 'Standard Customer'
        END AS Segment

    FROM dummycustomermaster
) AS source
ON target.CustomerID = source.CustomerID
WHEN MATCHED THEN
    UPDATE SET
        target.CustomerName = source.CustomerName,
        target.Segment = source.Segment,
        target.RiskBand = source.RiskBand,
        target.RegionID = source.RegionID,
        target.Phone = source.Phone,
        target.Email = source.Email
WHEN NOT MATCHED THEN
    INSERT (customersk, CustomerID, CustomerName, Segment, RiskBand, RegionID, Phone, Email)
    VALUES (source.customersk, source.CustomerID, source.CustomerName, source.Segment, source.RiskBand,
            source.RegionID, source.Phone, source.Email);
select * from dimcustomer_backup order by customerid;


In [ ]:
# create task_merge_dimproduct_hourly

CREATE OR REPLACE TASK task_merge_dimproduct_hourly
  WAREHOUSE = my_warehouse
  SCHEDULE = "1 minutes"
AS
MERGE INTO DIMPRODUCT_BACKUP AS target
USING (
    SELECT
        ROW_NUMBER() OVER (ORDER BY la.productid) AS productsk,
        la.PRODUCTID,

        -- Business Rule: Simple SECURED/UNSECURED based on CollateralID
        CASE
            WHEN COUNT(coll.COLLATERALID) > 0 THEN
                CASE
                    WHEN COUNT(DISTINCT coll.COLLATERALID) > 1 THEN 'MULTI_SECURED'
                    WHEN AVG(coll.valuation) > 200000 THEN 'OVER_SECURED'
                    WHEN AVG(coll.VALUATION) >= 350000 THEN 'FULLY_SECURED'
                    ELSE 'PARTIALLY_SECURED'
                END
            WHEN AVG(la.ORIGINATIONAMOUNT) > 50000 THEN 'LARGE_UNSECURED'
            WHEN AVG(la.ORIGINATIONAMOUNT) <= 10000 THEN 'SMALL_UNSECURED'
            ELSE 'STANDARD_UNSECURED'
        END AS LoanType,

        -- Business Rule: Tenor bands based on average months
        CASE
            WHEN AVG(la.TENORMONTHS) <= 3 THEN 'ULTRA_SHORT'
            WHEN AVG(la.TENORMONTHS) <= 6 THEN 'VERY_SHORT'
            WHEN AVG(la.TENORMONTHS) <= 12 THEN 'SHORT_TERM'
            WHEN AVG(la.TENORMONTHS) <= 24 THEN 'MEDIUM_TERM'
            WHEN AVG(la.TENORMONTHS) <= 36 THEN 'LONG_TERM'
            WHEN AVG(la.TENORMONTHS) <= 60 THEN 'VERY_LONG'
            ELSE 'EXTENDED_TERM'
        END AS Tenor,

        -- Business Rule: APR tiers
        CASE
            WHEN AVG(la.APR) < 3 THEN 'PRIME'
            WHEN AVG(la.APR) < 5 THEN 'PREFERRED'
            WHEN AVG(la.APR) < 7 THEN 'STANDARD'
            WHEN AVG(la.APR) < 10 THEN 'NON_PRIME'
            WHEN AVG(la.APR) < 15 THEN 'SUBPRIME'
            WHEN AVG(la.APR) < 20 THEN 'HIGH_RISK'
            ELSE 'SPECIALTY'
        END AS APRTier
    FROM dummyloan la
    LEFT JOIN dummycollateralregister coll ON la.LOANID = coll.LOANID
    GROUP BY la.PRODUCTID
) AS source
ON target.ProductID = source.ProductID
WHEN NOT MATCHED THEN
    INSERT (productsk, ProductID, LoanType, Tenor, APRTier)
    VALUES (source.productsk, source.ProductID, source.LoanType, source.Tenor, source.APRTier);

    select * from dimproduct_backup;




In [ ]:
# check

  -- ------------------------------------------------------- checking

desc table customer_master;

desc table collateral_register;
desc table collection_events;
desc table region_hierarchy;

create or replace table dummycustomermaster as select * from customer_master;
create or replace table dummycollateralregister as select * from collateral_register;
create or replace table dummyregionhierarchy as select * from region_hierarchy;
create or replace table dummycollectionevents as select * from collection_events;

-- --------------------------------------------------------------------

select * from dimcustomer_backup order by customerid;
select * from dimcollateral_backup order by collateralid;
select * from dimproduct_backup order by productid;
select * from dimregion_backup order by regionid;


In [ ]:
#

-- merge for factloan (not working yet)


MERGE INTO dummyFactLoan fact
USING (
    SELECT
        stg.loanid,
        -- Get surrogate keys from dimension tables
        cust.customersk,  -- From DimCustomer (surrogate key)
        pro.productsk,    -- From DimProduct (surrogate key)
        reg.regionsk,     -- From DimRegion (surrogate key)
        COALESCE(dcol.collateralsk, -1) AS collateralsk,  -- Get collateral key, if available
        stg.originationamount,
        stg.apr,
        stg.tenormonths AS tenor,
        stg.currentbalance,
        stg.status
    FROM dummyloan stg
    JOIN dimcustomer cust ON cust.customerid = stg.customerid
    JOIN dimregion reg ON reg.regionid = stg.regionid
    JOIN dimproduct pro ON pro.productid = stg.productid
    LEFT JOIN collateral_register coll ON coll.loanid = stg.loanid
    LEFT JOIN dimcollateral dcol ON dcol.collateralid = coll.collateralid
) src
ON fact.loanid = src.loanid  -- Match on loanid to update if exists

WHEN MATCHED AND (
    MD5(
        TO_VARCHAR(fact.customerkey) ||
        TO_VARCHAR(fact.regionkey) ||
        TO_VARCHAR(fact.productkey) ||
        TO_VARCHAR(fact.collateralkey) ||
        TO_VARCHAR(fact.originationamount) ||
        TO_VARCHAR(fact.apr) ||
        TO_VARCHAR(fact.tenor) ||
        TO_VARCHAR(fact.currentbalance) ||
        TO_VARCHAR(fact.status)
    ) !=
    MD5(
        TO_VARCHAR(src.customersk) ||
        TO_VARCHAR(src.regionsk) ||
        TO_VARCHAR(src.productsk) ||
        TO_VARCHAR(src.collateralsk) ||
        TO_VARCHAR(src.originationamount) ||
        TO_VARCHAR(src.apr) ||
        TO_VARCHAR(src.tenor) ||
        TO_VARCHAR(src.currentbalance) ||
        TO_VARCHAR(src.status)
    )
) THEN
    -- If any of the values have changed, update the fact table
    UPDATE SET
        fact.customerkey = src.customersk,
        fact.regionkey = src.regionsk,
        fact.productkey = src.productsk,
        fact.collateralkey = src.collateralsk,
        fact.originationamount = src.originationamount,
        fact.apr = src.apr,
        fact.tenor = src.tenor,
        fact.currentbalance = src.currentbalance,
        fact.status = src.status

WHEN NOT MATCHED THEN
    -- If no match found, insert new record with surrogate keys
    INSERT (
        loanid,
        customerkey,
        productkey,
        regionkey,
        collateralkey,
        originationamount,
        apr,
        tenor,
        currentbalance,
        status
    )
    VALUES (
        src.loanid,
        src.customersk,
        src.productsk,
        src.regionsk,
        src.collateralsk,
        src.originationamount,
        src.apr,
        src.tenor,
        src.currentbalance,
        src.status
    );


# **Fact collection task**

In [ ]:
# creation queries

CREATE OR REPLACE STREAM collection_events_stream
ON TABLE collection_events;

CREATE OR REPLACE TABLE audit_log (
    audit_id NUMBER(38,0),
    pipeline_name STRING,
    run_start_time TIMESTAMP,
    run_end_time TIMESTAMP,
    total_events_processed NUMBER,
    total_events_updated NUMBER,
    total_events_inserted NUMBER,
    status STRING
);


In [ ]:
# procedure query

CREATE OR REPLACE PROCEDURE sp_update_factcollections()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN

    CREATE OR REPLACE TEMP TABLE stream_buffer AS
    SELECT *
    FROM collection_events_stream;

    CREATE OR REPLACE TEMP TABLE src_data AS
    SELECT *
    FROM (
        SELECT
            ce.EVENTID,
            dc.CustomerSK,
            dr.RegionSK,
            ce.DPD,
            ce.PaymentAmount,
            ce.PTP_Flag AS PTP,
            CASE
                WHEN COALESCE(ce.PaymentAmount,0) > 0 THEN 'Paid'
                WHEN COALESCE(ce.PTP_Flag,0) = 1 AND COALESCE(ce.PaymentAmount,0) = 0 THEN 'Promise to Pay'
                WHEN COALESCE(ce.PaymentAmount,0) = 0 AND COALESCE(ce.PTP_Flag,0) = 0 THEN 'Not Connected'
                ELSE 'No Action'
            END AS ContactResult,
            ROUND(
                LEAST(
                    (SUM(ce.PaymentAmount) OVER (PARTITION BY ld.CustomerID) /
                     NULLIF(SUM(ld.CurrentBalance) OVER (PARTITION BY ld.CustomerID),0)) * 100,
                    100
                ), 2
            ) AS CurePercent,
            ROW_NUMBER() OVER (PARTITION BY ce.EVENTID ORDER BY ce.EVENTDATE DESC) AS rn
        FROM stream_buffer ce
        LEFT JOIN loan_data ld ON ce.LoanID = ld.LoanID
        LEFT JOIN dimcustomer dc ON ld.CustomerID = dc.CustomerID
        LEFT JOIN dimregion dr ON ld.RegionID = dr.RegionID
    )
    WHERE rn = 1;

    MERGE INTO factcollections tgt
    USING src_data src
    ON tgt.EVENTID = src.EVENTID
    WHEN MATCHED THEN
        UPDATE SET
            DPD = src.DPD,
            ContactResult = src.ContactResult,
            PTP = src.PTP,
            CurePercent = src.CurePercent
    WHEN NOT MATCHED THEN
        INSERT (EVENTID, CustomerSK, RegionSK, DPD, ContactResult, PTP, CurePercent)
        VALUES (src.EVENTID, src.CustomerSK, src.RegionSK, src.DPD, src.ContactResult, src.PTP, src.CurePercent);

    INSERT INTO audit_log (
        audit_id,
        pipeline_name,
        run_start_time,
        run_end_time,
        total_events_processed,
        total_events_updated,
        total_events_inserted,
        status
    )
    WITH agg AS (
        SELECT
            COUNT(*) AS total_events_processed,
            SUM(CASE WHEN tgt.EVENTID IS NOT NULL THEN 1 ELSE 0 END) AS total_events_updated,
            SUM(CASE WHEN tgt.EVENTID IS NULL THEN 1 ELSE 0 END) AS total_events_inserted
        FROM stream_buffer ce
        LEFT JOIN factcollections tgt ON ce.EVENTID = tgt.EVENTID
    ),
    next_id AS (
        SELECT COALESCE(MAX(audit_id), 0) AS last_id FROM audit_log
    )
    SELECT
        last_id + 1 AS audit_id,
        'FactCollections_Update',
        CURRENT_TIMESTAMP,
        CURRENT_TIMESTAMP,
        total_events_processed,
        total_events_updated,
        total_events_inserted,
        'SUCCESS'
    FROM agg
    CROSS JOIN next_id;

    RETURN 'FactCollections Update Completed';
END;
$$;


In [ ]:
# task query

CREATE OR REPLACE TASK task_update_factcollections
  WAREHOUSE = my_warehouse
  SCHEDULE = 'USING CRON 0 * * * * UTC'
  -- schedule = '1 minute'
AS
CALL sp_update_factcollections();

ALTER TASK my_schema.task_update_factcollections resume;
ALTER TASK my_schema.task_update_factcollections suspend;

In [ ]:
# testing queries

show tasks like '%task_update_factcollections';
show tasks;

select * from collection_events where EVENTID=20000000;

--  update an existing event
UPDATE collection_events
SET
    DPD = 70,
    ACTIONTYPE = 'Call',
    PTP_FLAG = 1,
    PAYMENTAMOUNT = 445.9,
    AGENTID = 1034,
    EVENTDATE = '2020-10-25'
WHERE EVENTID = 20000000;


select * from collection_events;
select * from factcollections where EVENTID=20000000;

select * from audit_log;
-- truncate table audit_log;
select * from  collection_events_stream;

# update result
# 6424	68	30.00	Paid	1.00	2.35	20000000
# 6424	68	70.00	Paid	1.00	3.68	20000000

## Task 10: Materialized Views for Performance


• mv_vintage_performance: Aggregates loss ratio by origination month.
• mv_roll_rate: Tracks transitions between delinquency buckets.
• mv_collections_effectiveness: Summarizes PTP kept %, cure rates.


In [ ]:
%%sql

create or replace table for_mv_collections_effectiveness as
SELECT
    fl.LoanID,
    ce.eventid,
    curepercent,
    COUNT(DISTINCT CASE WHEN fc.PTP = 1 THEN ce.EventID END) AS PromisesMade,
    COUNT(DISTINCT CASE WHEN fc.PTP = 1 AND ce.PaymentAmount > 0 THEN ce.EventID END) AS PromisesKept,
    ROUND(
        (COUNT(DISTINCT CASE WHEN  fc.PTP = 1 AND ce.PaymentAmount > 0 THEN ce.EventID END) * 100.0) /
        NULLIF(COUNT(DISTINCT CASE WHEN fc.PTP = 1 THEN ce.EventID END), 0),
        2
    ) AS PTP_Percentage
FROM FactCollections fc join factloan fl on fl.customerkey = fc.customersk join collection_events ce on ce.loanid = fl.loanid
GROUP BY fl.LoanID, curepercent, ce.eventid
having curepercent > 0
ORDER BY fl.LoanID;
select * from for_mv_collections_effectiveness where promisesmade != promiseskept and promiseskept !=0;

create materialized view mv_collections_effectiveness
as
select loanid, eventid, curepercent, ptp_percentage
from for_mv_collections_effectiveness;

select * from mv_collections_effectiveness ;

SyntaxError: invalid syntax (ipython-input-3139531607.py, line 1)

In [ ]:
#mv_vintage_performance


SELECT
    COUNT(*) AS loan_count,
    SUM(ORIGINATIONAMOUNT) AS total_origination_amount,
    SUM(CASE
            WHEN ORIGINATIONAMOUNT - CURRENTBALANCE > 0
            THEN ORIGINATIONAMOUNT - CURRENTBALANCE
            ELSE 0
        END) AS total_loss_amount,
    CASE
        WHEN SUM(ORIGINATIONAMOUNT) = 0 THEN 0
        ELSE SUM(CASE
                     WHEN ORIGINATIONAMOUNT - CURRENTBALANCE > 0
                     THEN ORIGINATIONAMOUNT - CURRENTBALANCE
                     ELSE 0
                 END)
             / SUM(ORIGINATIONAMOUNT)
    END AS loss_ratio
FROM factloan;

SELECT * FROM mv_vintage_performance;



select * from factloan;
select * from factloan;
drop table factloan1;



# **PHASE 4: 🔐 Security, Governance and Sharing**




## Task 11: Data Masking and RLS


• Apply MASKING POLICY on:
 o DimCustomer.customer_name
 o DimCustomer.email
 o DimCustomer.phone
• Implement ROW ACCESS POLICY restricting portfolio access by Region or Country.


In [ ]:
------------------MASKING IN NAME---------------------
--MASKING IN DIMCUSTOMER
CREATE OR REPLACE MASKING POLICY mask_customer_name AS (CUSTOMERNAME STRING)
RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN') THEN CUSTOMERNAME
    ELSE
      -- Extract number after underscore
      CASE
        WHEN TRY_TO_NUMBER(SPLIT_PART(CUSTOMERNAME, '_', 2)) >= 1000
          THEN CONCAT('XXXXXXX_', TO_VARCHAR(FLOOR(TRY_TO_NUMBER(SPLIT_PART(CUSTOMERNAME, '_', 2)) / 1000)) || 'K')
        ELSE
          CONCAT('XXXXXXX_', SPLIT_PART(CUSTOMERNAME, '_', 2))
      END
  END;
  ALTER TABLE DimCustomer MODIFY COLUMN CUSTOMERNAME SET MASKING POLICY mask_customer_name;
    ALTER TABLE DimCustomer MODIFY COLUMN CUSTOMERNAME UNSET MASKING POLICY ;

------------------MASKING IN EMAIL---------------------
CREATE OR REPLACE MASKING POLICY mask_email AS (EMAIL STRING)
RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN') THEN EMAIL
    ELSE 'xxxx@gmail.com'
  END;
ALTER TABLE DimCustomer MODIFY COLUMN EMAIL SET MASKING POLICY mask_email;
SELECT*FROM DIMCUSTOMER;

------------------MASKING IN PHONE---------------------
CREATE OR REPLACE MASKING POLICY mask_phone_random AS (PHONE STRING)
RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN') THEN PHONE
    ELSE
      CONCAT('(91)XXX-XXX-XX',
             LPAD(TO_VARCHAR(UNIFORM(10, 99, RANDOM())), 2, '0'))
  END;
ALTER TABLE DimCustomer MODIFY COLUMN PHONE SET MASKING POLICY mask_phone_random;
ALTER WAREHOUSE MY_WAREHOUSE SUSPEND;
SELECT*FROM DIMCUSTOMER;

In [ ]:
-----------------ROW ACCESS POLICY-------------------------
CREATE OR REPLACE ROW ACCESS POLICY COUNTRY_ROLE AS (COUNTRY STRING)
RETURNS BOOLEAN ->
CASE
WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN TRUE
WHEN CURRENT_ROLE()='CAN_ROLE' AND COUNTRY='Canada' THEN TRUE
WHEN CURRENT_ROLE()='FRANCE_ROLE' AND COUNTRY='France' THEN TRUE
WHEN CURRENT_ROLE()='GERMANY_ROLE' AND COUNTRY='Germany' THEN TRUE
WHEN CURRENT_ROLE()='INDIA_ROLE' AND COUNTRY='India' THEN TRUE
WHEN CURRENT_ROLE()='UK_ROLE' AND COUNTRY='UK' THEN TRUE
WHEN CURRENT_ROLE()='USA_ROLE' AND COUNTRY='USA' THEN TRUE
ELSE FALSE
END;

ALTER TABLE DIMREGION ADD ROW ACCESS POLICY COUNTRY_ROLE ON (COUNTRY);
ALTER TABLE DIMREGION DROP ROW ACCESS POLICY COUNTRY_ROLE;
SELECT * FROM DIMREGION;

## Task 12: Secure Data Sharing



• Create SECURE SHARE exposing:
 o mv_vintage_performance
 o mv_collections_effectiveness
• Test accessibility via reader account.
• Document sharing process and validation.



In [ ]:
CREATE SHARE SECURE_view_share;

GRANT USAGE ON DATABASE my_database TO SHARE SECURE_view_share;
GRANT USAGE ON SCHEMA my_database.my_schema TO SHARE SECURE_view_share;
GRANT SELECT ON TABLE my_database.my_schema.mv_collections_effectiveness TO SHARE SECURE_view_share;
GRANT SELECT ON TABLE my_database.my_schema.mv_vintage_performance TO SHARE SECURE_view_share;

ALTER SHARE secure_view_share ADD ACCOUNTS = JTC04550;
show managed accounts;
ALTER SHARE secure_view_share SET SECURE_OBJECTS_ONLY = FALSE;


CREATE MANAGED ACCOUNT reader_account
ADMIN_NAME = 'reader_admin'
ADMIN_PASSWORD = 'Admin@12345678'
TYPE = READER
COMMENT = 'Reader account for external partner';
SHOW MANAGED ACCOUNTS;

CREATE OR REPLACE MATERIALIZED VIEW mv_collections_ptp_kept AS
SELECT
    AgentID,
    COUNT(DISTINCT CASE WHEN fc.PTP = 1 THEN ce.EventID END) AS PromisesMade,
    COUNT(DISTINCT CASE WHEN fc.PTP = 1 AND ce.PaymentAmount > 0 THEN ce.EventID END) AS PromisesKept,
    ROUND(
        (COUNT(DISTINCT CASE WHEN fc.PTP = 1 AND ce.PaymentAmount > 0 THEN ce.EventID END) / NULLIF(COUNT(DISTINCT CASE WHEN fc.PTP = 1 THEN ce.EventID END), 0)) * 100,
        2
    ) AS PTP_Kept_Percentage
FROM stg_collections_events
GROUP BY AgentID
ORDER BY AgentID;







